<a href="https://colab.research.google.com/github/Khushibung05/RAG/blob/main/chunking_Embedding_vectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Chunking**

In [ ]:
text = """
Employees receive 12 casual leaves annually.
Employees can work from home twice per week.
Travel expenses are reimbursed within 30 days.
Medical insurance is provided to all employees.
"""

###Fixed-size chunking

In [ ]:
chunk_size=50
chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
for chunk in chunks:
  print(chunk)
  print('-'*50)


Employees receive 12 casual leaves annually.
Empl
--------------------------------------------------
oyees can work from home twice per week.
Travel ex
--------------------------------------------------
penses are reimbursed within 30 days.
Medical insu
--------------------------------------------------
rance is provided to all employees.

--------------------------------------------------


i.e., texts is divided into chunks each having 50 chars

###Recursive chunking

In [ ]:
!pip install -q langchain langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)
chunks=splitter.split_text(text)
for i,chunk in enumerate(chunks):
  print(f"chunk {i+1}")
  print(chunk)
  print('-'*50)

chunk 1
Employees receive 12 casual leaves annually.
--------------------------------------------------
chunk 2
Employees can work from home twice per week.
--------------------------------------------------
chunk 3
Travel expenses are reimbursed within 30 days.
--------------------------------------------------
chunk 4
Medical insurance is provided to all employees.
--------------------------------------------------


###Basic sentence chunking using NLTK

In [ ]:
!pip install nltk

In [ ]:
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab') # Changed to punkt_tab as per the error message
#split into sentence
chunks=sent_tokenize(text)
for i,chunk in enumerate(chunks):
  print(f"chunk {i+1}:{chunk}")
  print('-'*50)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


chunk 1:
Employees receive 12 casual leaves annually.
--------------------------------------------------
chunk 2:Employees can work from home twice per week.
--------------------------------------------------
chunk 3:Travel expenses are reimbursed within 30 days.
--------------------------------------------------
chunk 4:Medical insurance is provided to all employees.
--------------------------------------------------


###Group 2 Sentences per chunk

In [ ]:
sentences=sent_tokenize(text)
chunk_size=2
chunks=[]
for i in range(0,len(sentences),chunk_size):
  chunk=' '.join(sentences[i:i+chunk_size])
  chunks.append(chunk)
for i,chunk in enumerate(chunks,start=1):
  print(f"\nchunk {i}:{chunk}")


chunk 1:
Employees receive 12 casual leaves annually. Employees can work from home twice per week.

chunk 2:Travel expenses are reimbursed within 30 days. Medical insurance is provided to all employees.


###Semantic Chunking using sentence transformers

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
sentences=sent_tokenize(text)
model=SentenceTransformer('all-MiniLM-L6-v2')
embeddings=model.encode(sentences)
threshold=0.5
chunks=[]
current_chunk=[sentences[0]]
for i in range(1,len(sentences)):
  similarity=cosine_similarity([embeddings[i-1]],[embeddings[i]])[0][0]
  if similarity>threshold:
    current_chunk.append(sentences[i])
  else:
    chunks.append(' '.join(current_chunk))
    current_chunk=[sentences[i]]
chunks.append(' '.join(current_chunk))

for id,chunk in enumerate(chunks,start=1):
  print(f"chunk {i}:{chunk}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

chunk 3:
Employees receive 12 casual leaves annually.
chunk 3:Employees can work from home twice per week.
chunk 3:Travel expenses are reimbursed within 30 days.
chunk 3:Medical insurance is provided to all employees.


###**Embeddings**

In [ ]:
document = """
Leave Policy

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.

Work From Home Policy

Employees may work from home twice per week.
Manager approval is required for additional remote work.

Travel Policy

Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.

Medical Insurance Policy

All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
"""
print("="*50)
print(document)
print("="*50)


Leave Policy
 
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
 
Work From Home Policy
 
Employees may work from home twice per week.
Manager approval is required for additional remote work.
 
Travel Policy
 
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
 
Medical Insurance Policy
 
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.



In [ ]:
#Recursive chunks
splitter=RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)
chunks=splitter.split_text(document)

print("\n")
print("="*80)
print("Generated chunks")
print("="*80)
for i,chunk in enumerate(chunks,start=1):
  print(f"chunk {i}:{chunk}")
  print('-'*50)



Generated chunks
chunk 1:Leave Policy
 
Employees receive 12 casual leaves annually.
--------------------------------------------------
chunk 2:Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
--------------------------------------------------
chunk 3:Work From Home Policy
 
Employees may work from home twice per week.
--------------------------------------------------
chunk 4:Manager approval is required for additional remote work.
 
Travel Policy
--------------------------------------------------
chunk 5:Travel Policy
 
Travel expenses are reimbursed within 30 days.
--------------------------------------------------
chunk 6:Original receipts must be submitted for reimbursement.
 
Medical Insurance Policy
--------------------------------------------------
chunk 7:All employees are covered under company medical insurance.
--------------------------------------------------
chunk 8:Dependent coverage is available for spouse and children.
------

###Load Embedding Model
all-MiniLM-L6-v2

output vector Dimension=384

In [ ]:
model=SentenceTransformer("all-MiniLM-L6-v2")

print("\n Embedding model loadedsuccessfully")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


 Embedding model loadedsuccessfully


###Geenerating Embeddings

In [ ]:
chunk_embeddings=model.encode(chunks)
print("\n Embedding generated.")


 Embedding generated.


###Display Embedding information

In [ ]:
print("\n")
print("="*50)
print("Embedding Details")
print("="*50)

print("No. of chunks:",len(chunks))
print("Embedding shape:",chunk_embeddings.shape)



Embedding Details
No. of chunks: 8
Embedding shape: (8, 384)


In [ ]:
print("="*50)
print("First Chunk")
print("="*50)
print(chunks[0])


First Chunk
Leave Policy
 
Employees receive 12 casual leaves annually.


In [ ]:
print("="*50)
print("First 20 embedding values")
print("="*50)
print(chunk_embeddings[0][:20])


First 20 embedding values
[ 0.06098088  0.0266781   0.0129045   0.02433847  0.09787528  0.09084512
  0.00728962 -0.03919934 -0.05303932  0.02543474  0.09522852  0.04315791
 -0.04351876 -0.03531914 -0.00686681  0.04278306 -0.06461596 -0.02985374
 -0.00127592 -0.06867143]


In [ ]:
print("="*50)
print("chunk to vector mapping")
print("="*50)
for i,chunk in enumerate(chunks):
  print(f"chunk {i+1}:{chunk[:20]}")
  print("\nvector shape:")
  print(chunk_embeddings[i].shape)
  print('-'*50)

chunk to vector mapping
chunk 1:Leave Policy
 
Emplo

vector shape:
(384,)
--------------------------------------------------
chunk 2:Employees receive 15

vector shape:
(384,)
--------------------------------------------------
chunk 3:Work From Home Polic

vector shape:
(384,)
--------------------------------------------------
chunk 4:Manager approval is 

vector shape:
(384,)
--------------------------------------------------
chunk 5:Travel Policy
 
Trav

vector shape:
(384,)
--------------------------------------------------
chunk 6:Original receipts mu

vector shape:
(384,)
--------------------------------------------------
chunk 7:All employees are co

vector shape:
(384,)
--------------------------------------------------
chunk 8:Dependent coverage i

vector shape:
(384,)
--------------------------------------------------


###**vector DB**

In [ ]:
chunks = [

    "Employees receive 12 casual leaves annually.",

    "Employees receive 15 sick leaves annually.",

    "Employees may work from home twice per week.",

    "Travel expenses are reimbursed within 30 days.",

    "All employees are covered under company medical insurance."

]

print("Total Chunks:", len(chunks))


Total Chunks: 5


###loAD embedding model

In [ ]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")
print("Embedded model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedded model loaded


###Generate embeddings

In [ ]:
embeddings=model.encode(chunks)
print("Embedding shape:")
print(embeddings.shape)

Embedding shape:
(5, 384)


####5 chunks,384 dimensional vectors since MiniLm model is used

###Create FAISS index

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 47.1 MB/s eta 0:00:00


In [ ]:
import faiss
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
print("faiss index created")

faiss index created


####Add Embedding to FAISS

In [ ]:
import numpy as np
index.add(
    np.array(embeddings,dtype=np.float32)
)
print("vectors stored:",index.ntotal)

vectors stored: 5


###User query

In [ ]:
query="How many casual leaves are allowed"

###Convert Query into Embeddings

In [ ]:
query_embedding=model.encode(
    [query]
)

###Search FAISS

In [ ]:
k=3
distances,indices=index.search(
    np.array(
        query_embedding,
        dtype=np.float32
),k)

###Display Results

In [ ]:
print("Search Results:")
print("="*50)
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"Rank {rank+1}:")
    print(f"  Distance: {dist:.4f}")
    print(f"  Chunk Index: {idx}")
    print(f"  Chunk Content: {chunks[idx]}")
    print('-'*50)

Search Results:
Rank 1:
  Distance: 0.6736
  Chunk Index: 0
  Chunk Content: Employees receive 12 casual leaves annually.
--------------------------------------------------
Rank 2:
  Distance: 1.1913
  Chunk Index: 1
  Chunk Content: Employees receive 15 sick leaves annually.
--------------------------------------------------
Rank 3:
  Distance: 1.6336
  Chunk Index: 2
  Chunk Content: Employees may work from home twice per week.
--------------------------------------------------
